# Healthcare Call Center NLP Analytics

This notebook demonstrates an end-to-end NLP pipeline for analyzing healthcare call center transcripts:
- Sentiment analysis
- Complaint categorization
- Friction point detection
- Repeat caller analysis
- KPI dashboard visualizations

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.sentiment import add_sentiment_columns
from src.categorization import add_predicted_category, build_tfidf_features
from src.friction import add_friction_columns
from src.repeat_callers import add_repeat_caller_flags, repeat_caller_summary, compute_fcr
from src.preprocessing import preprocess_column

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/synthetic_call_transcripts.csv')
print(f'Total records: {len(df)}')
print(f'Unique members: {df["member_id"].nunique()}')
print(f'Date range: {df["call_date"].min()} to {df["call_date"].max()}')
df.head(3)

## 2. Text Preprocessing

In [ ]:
df['transcript_clean'] = preprocess_column(df['transcript'])
print('Sample cleaned transcript:')
print(df['transcript_clean'].iloc[0][:200])

## 3. Sentiment Analysis

In [ ]:
df = add_sentiment_columns(df)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution of compound scores
axes[0].hist(df['sentiment_compound'], bins=30, edgecolor='black', alpha=0.7, color='#4C72B0')
axes[0].set_title('Distribution of Sentiment Scores', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Compound Score')
axes[0].set_ylabel('Count')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)

# Sentiment class breakdown
sentiment_counts = df['sentiment_class'].value_counts()
colors = {'negative': '#E74C3C', 'neutral': '#95A5A6', 'positive': '#27AE60'}
axes[1].bar(sentiment_counts.index, sentiment_counts.values,
            color=[colors[c] for c in sentiment_counts.index], edgecolor='black')
axes[1].set_title('Sentiment Classification', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Calls')

# Sentiment by category
cat_sentiment = df.groupby('category')['sentiment_compound'].mean().sort_values()
bar_colors = ['#E74C3C' if v < -0.05 else '#27AE60' if v > 0.05 else '#95A5A6' for v in cat_sentiment.values]
axes[2].barh(cat_sentiment.index, cat_sentiment.values, color=bar_colors, edgecolor='black')
axes[2].set_title('Avg Sentiment by Category', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Mean Compound Score')
axes[2].axvline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\nOverall sentiment distribution:")
print(df['sentiment_class'].value_counts(normalize=True).round(3))

## 4. Call Categorization

In [ ]:
df = add_predicted_category(df)

# Confusion-style accuracy check
accuracy = (df['category'] == df['predicted_category']).mean()
print(f'Keyword categorization accuracy vs ground truth: {accuracy:.1%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Call volume by category
cat_counts = df['category'].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, color='#4C72B0', edgecolor='black')
axes[0].set_title('Call Volume by Category', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Calls')
axes[0].tick_params(axis='x', rotation=15)

# Resolution rate by category
resolution = df.groupby('category')['resolved'].mean().sort_values()
bar_colors = ['#E74C3C' if v < 0.5 else '#F39C12' if v < 0.65 else '#27AE60' for v in resolution.values]
axes[1].barh(resolution.index, resolution.values, color=bar_colors, edgecolor='black')
axes[1].set_title('Resolution Rate by Category', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Resolution Rate')
axes[1].set_xlim(0, 1)
for i, v in enumerate(resolution.values):
    axes[1].text(v + 0.02, i, f'{v:.0%}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. TF-IDF Top Terms by Category

In [ ]:
tfidf_matrix, vectorizer = build_tfidf_features(df['transcript_clean'])
feature_names = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=False)

for idx, cat in enumerate(df['category'].unique()):
    mask = df['category'] == cat
    cat_tfidf = tfidf_matrix[mask].mean(axis=0).A1
    top_indices = cat_tfidf.argsort()[-10:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    top_scores = [cat_tfidf[i] for i in top_indices]
    
    axes[idx].barh(range(len(top_terms)), top_scores, color='#4C72B0', edgecolor='black')
    axes[idx].set_yticks(range(len(top_terms)))
    axes[idx].set_yticklabels(top_terms, fontsize=9)
    axes[idx].set_title(cat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[idx].invert_yaxis()

plt.suptitle('Top TF-IDF Terms by Call Category', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Friction Point Detection

In [ ]:
df = add_friction_columns(df)

friction_cols = [c for c in df.columns if c.startswith('friction_') and c != 'friction_count']
friction_rates = df[friction_cols].mean().sort_values(ascending=True)
friction_rates.index = [c.replace('friction_', '').replace('_', ' ').title() for c in friction_rates.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Friction type prevalence
axes[0].barh(friction_rates.index, friction_rates.values, color='#E74C3C', edgecolor='black', alpha=0.8)
axes[0].set_title('Friction Point Prevalence', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Proportion of Calls')
for i, v in enumerate(friction_rates.values):
    axes[0].text(v + 0.005, i, f'{v:.1%}', va='center')

# Friction vs sentiment
friction_groups = df.groupby('has_friction')['sentiment_compound'].mean()
labels = ['No Friction', 'Has Friction']
colors = ['#27AE60', '#E74C3C']
axes[1].bar(labels, friction_groups.values, color=colors, edgecolor='black')
axes[1].set_title('Avg Sentiment: Friction vs No Friction', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Mean Compound Score')
for i, v in enumerate(friction_groups.values):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nCalls with friction: {df["has_friction"].sum()} ({df["has_friction"].mean():.1%})')
print(f'Avg friction types per affected call: {df.loc[df["has_friction"], "friction_count"].mean():.1f}')

## 7. Repeat Caller Analysis

In [ ]:
df = add_repeat_caller_flags(df)
repeat_summary = repeat_caller_summary(df)
fcr = compute_fcr(df)

print(f'First Contact Resolution (FCR): {fcr:.1%}')
print(f'Repeat callers: {df["is_repeat_caller"].any() and df.loc[df["is_repeat_caller"], "member_id"].nunique()} members')
print(f'Repeat caller rate: {df["is_repeat_caller"].any() and df.loc[df["is_repeat_caller"], "member_id"].nunique() / df["member_id"].nunique():.1%}')
print(f'\nTop 10 repeat callers:')
repeat_summary.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Repeat vs one-time: sentiment comparison
repeat_labels = ['One-time', 'Repeat']
repeat_sentiments = [df.loc[~df['is_repeat_caller'], 'sentiment_compound'].mean(),
                     df.loc[df['is_repeat_caller'], 'sentiment_compound'].mean()]
colors = ['#27AE60', '#E74C3C']
axes[0].bar(repeat_labels, repeat_sentiments, color=colors, edgecolor='black')
axes[0].set_title('Sentiment: Repeat vs One-time Callers', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Compound Score')
for i, v in enumerate(repeat_sentiments):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Distribution of call counts for repeat callers
call_dist = repeat_summary['total_calls'].value_counts().sort_index()
axes[1].bar(call_dist.index, call_dist.values, color='#4C72B0', edgecolor='black')
axes[1].set_title('Call Frequency Distribution (Repeat Callers)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Calls')
axes[1].set_ylabel('Number of Members')

# Primary category for repeat callers
repeat_cats = repeat_summary['primary_category'].value_counts()
axes[2].pie(repeat_cats.values, labels=repeat_cats.index, autopct='%1.0f%%',
            colors=sns.color_palette('muted', len(repeat_cats)), startangle=90)
axes[2].set_title('Repeat Callers by Primary Category', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Executive KPI Summary

In [ ]:
total_calls = len(df)
unique_members = df['member_id'].nunique()
repeat_rate = df.loc[df['is_repeat_caller'], 'member_id'].nunique() / unique_members
overall_resolution = df['resolved'].mean()
avg_duration = df['duration_minutes'].mean()
negative_pct = (df['sentiment_class'] == 'negative').mean()
friction_pct = df['has_friction'].mean()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Healthcare Call Center — Executive KPI Dashboard', fontsize=16, fontweight='bold', y=1.02)

kpis = [
    ('Total Calls', f'{total_calls:,}', '#3498DB'),
    ('Unique Members', f'{unique_members:,}', '#2ECC71'),
    ('FCR Rate', f'{fcr:.0%}', '#F39C12' if fcr < 0.7 else '#27AE60'),
    ('Repeat Caller Rate', f'{repeat_rate:.0%}', '#E74C3C' if repeat_rate > 0.25 else '#F39C12'),
    ('Resolution Rate', f'{overall_resolution:.0%}', '#27AE60' if overall_resolution > 0.6 else '#E74C3C'),
    ('Avg Duration (min)', f'{avg_duration:.1f}', '#3498DB'),
    ('Negative Sentiment', f'{negative_pct:.0%}', '#E74C3C' if negative_pct > 0.3 else '#F39C12'),
    ('Friction Rate', f'{friction_pct:.0%}', '#E74C3C' if friction_pct > 0.3 else '#F39C12'),
]

for idx, (label, value, color) in enumerate(kpis):
    row, col = divmod(idx, 4)
    ax = axes[row][col]
    ax.text(0.5, 0.55, value, transform=ax.transAxes, fontsize=32, fontweight='bold',
            ha='center', va='center', color=color)
    ax.text(0.5, 0.2, label, transform=ax.transAxes, fontsize=12,
            ha='center', va='center', color='#555555')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, edgecolor='#DDDDDD', linewidth=2, transform=ax.transAxes))

plt.tight_layout()
plt.show()

## 9. Key Findings & Recommendations

### Findings
1. **Billing & Claims dominate call volume** — these two categories account for the majority of contacts and have the lowest satisfaction scores
2. **Repeat callers show significantly worse sentiment** — members who call back are frustrated, suggesting first-contact resolution gaps
3. **Friction points correlate strongly with negative sentiment** — long holds and multiple transfers are the top drivers
4. **FCR gap** — the first-contact resolution rate indicates room for improvement, particularly in billing and claims

### Recommendations
1. **Invest in billing/claims self-service tools** — reduce call volume by enabling online claim status and billing dispute resolution
2. **Implement callback technology** — reduce hold-time friction by offering scheduled callbacks
3. **Deploy call routing improvements** — reduce transfers by better matching call reasons to specialized agents
4. **Create repeat-caller intervention program** — proactively reach out to members with 3+ contacts to resolve root causes